# Post-training a VLM for reasoning with GRPO using TRL

_Authored by: [Sergio Paniego](https://github.com/sergiopaniego)_

🚨 **WARNING**: This section requires substantial computational resources. A Colab run uses an A100 GPU.

This section shows how to post-train a [Vision Language Model (VLM)](https://huggingface.co/blog/vlms-2025) with [GRPO](https://huggingface.co/docs/trl/grpo_trainer). The implementation uses the Hugging Face [Transformer Reinforcement Learning library (trl)](https://huggingface.co/docs/trl/index).


We fine-tune [Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct) on a subset of the [lmms-lab/multimodal-open-r1-8k-verified](https://huggingface.co/datasets/lmms-lab/multimodal-open-r1-8k-verified) dataset. Each example contains an image, a problem description, a solution, and a reasoning trace. We use this data format and the GRPO reward functions to train the model to reason toward the solution.

![Post-training a vision-language model with GRPO using TRL](images/vlm-grpo-trl-pipeline.png)

## 1. Install dependencies

Install the libraries required for fine-tuning.
Install `trl` from source because, at the time of writing, the official release does not include the VLM GRPO trainer.


In [ ]:
# Container already has trl, peft (>=0.19), torch. Extra deps (math_verify, qwen-vl-utils) from pyproject notebooks extras.
import sys
print(f"Python: {sys.version}")


Authenticate with your Hugging Face 🤗 account so you can save and share the trained model.

In [ ]:
import os
from huggingface_hub import whoami

try:
    info = whoami()
    name = info.get("fullname") or info.get("name", "unknown")
    print(f"Logged in as: {name}")
except Exception:
    if "HF_TOKEN" in os.environ:
        print("HF_TOKEN found in environment")
    else:
        print("WARNING: HF_TOKEN not set")


## 2. Load the dataset 📁

This section uses [lmms-lab/multimodal-open-r1-8k-verified](https://huggingface.co/datasets/lmms-lab/multimodal-open-r1-8k-verified). The dataset contains 8,000 multimodal reinforcement learning examples for mathematical reasoning. GPT-4o generated the data. Each sample includes `image`, `problem`, `solution`, `original question`, and `original answer`. The data comes from [this project](https://github.com/EvolvingLMMs-Lab/open-r1-multimodal).

We use `image` and `problem` as inputs and `solution` as the output so the model learns to reason from images.

To reduce training time, we use 5% of the dataset and split it into training and test sets. A full training run would use the complete dataset.

Load and split the dataset.

In [ ]:
from datasets import load_dataset

dataset_id = 'lmms-lab/multimodal-open-r1-8k-verified'
dataset = load_dataset(dataset_id, split='train[:5%]')

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

Inspect the dataset structure.

In [ ]:
print(train_dataset)

Inspect one sample:

In [ ]:
print(train_dataset[0])

In addition to the `problem` and `image` columns, we include a custom system prompt that specifies the output format.

The system prompt is adapted from DeepSeek R1. See [this previous recipe](https://huggingface.co/learn/cookbook/fine_tuning_llm_grpo_trl) for details.

We convert each dataset sample to the conversation format expected by the GRPO trainer. Each conversation contains the system prompt, one image, and one problem description.

We set `padding_side="left"` so each generated completion follows its prompt directly during training. GRPO requires this arrangement to compare token-level probabilities between preferred and rejected responses.

In [ ]:
from transformers import AutoProcessor

model_id = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_id, use_fast=True, padding_side="left")

SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

def make_conversation(example):
    # Return raw conversational prompt - TRL's GRPOTrainer applies the chat template internally
    return {
        "prompt": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": example["problem"]},
                ],
            },
        ],
        "image": example["image"],
    }

train_dataset = train_dataset.map(make_conversation)

Inspect a converted example:

In [ ]:
print(train_dataset[0]['prompt'])

Remove the columns that are not needed for training.

In [ ]:
train_dataset

Verify that the columns were removed.

In [ ]:
train_dataset = train_dataset.remove_columns(['problem', 'original_question', 'original_answer'])
print(train_dataset)

## 3. Post-training the VLM with GRPO

The diagram below shows the main difference between **PPO** (Proximal Policy Optimization) and **GRPO** (Group Relative Policy Optimization): GRPO does not use a value model. See this [further explanation](https://www.philschmid.de/deepseek-r1) for more detail.

The training pipeline uses trl, the Hugging Face reinforcement learning library. We use the `GRPOConfig` and `GRPOTrainer` classes. Custom reward functions specify the behavior required by the training objectives.

First, load [Qwen/Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen/Qwen2.5-VL-3B-Instruct), a VLM developed by [Qwen](https://huggingface.co/Qwen). A model with more parameters may produce better results.

Other VLM projects with reasoning capabilities include:


* [GLM-4.1V-9B-Thinking](https://huggingface.co/THUDM/GLM-4.1V-9B-Thinking)

* [VLM-R1 models](https://huggingface.co/collections/omlab/vlm-r1-models-67b7352db15c19d57157c348)

* [R1-V](https://github.com/StarsfieldAI/R1-V)

![PPO compared with GRPO: GRPO drops the value model and scores a group of sampled answers](images/ppo-vs-grpo.jpeg)

### 3.1 Load the baseline model

Load the `Qwen/Qwen2.5-VL-3B-Instruct` baseline model.


In [ ]:
import torch
from transformers import AutoModelForImageTextToText

model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

### 3.2 Configure LoRA

Configure LoRA for training.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

### 3.3 Load the reward functions



The reward can come from a pretrained reward model or from functions defined in code. The DeepSeek-R1 authors used an accuracy-based reward to evaluate whether a response is correct. They also used a format-based reward to require reasoning inside `<think> </think>` tags. See the [Open R1 implementation](https://github.com/huggingface/open-r1/blob/main/src/open_r1/rewards.py) for details. Here, we implement the rewards as Python functions.

We use the following reward functions from the Open R1 [implementation](https://github.com/huggingface/open-r1/blob/main/src/open_r1/rewards.py):

1. **Format enforcement:** Requires the generation to use `<think> </think> <answer> </answer>` tags for reasoning.  


In [ ]:
import re
def _extract_text(c):
    """Extract text from a completion which may be a string or list of messages."""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        parts = []
        for m in c:
            content = m.get("content") if isinstance(m, dict) else m
            if isinstance(content, str):
                parts.append(content)
            elif isinstance(content, list):
                for el in content:
                    if isinstance(el, dict) and "text" in el:
                        parts.append(el["text"])
        return "".join(parts)
    return str(c)


def format_reward(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>\n.*?\n</think>\n<answer>\n.*?\n</answer>$"
    texts = [_extract_text(c) for c in completions]
    matches = [re.match(pattern, t, re.DOTALL | re.MULTILINE) for t in texts]
    return [1.0 if m else 0.0 for m in matches]

2. **Solution accuracy:** Compares the generated solution with the `solution` column in the dataset.

In [ ]:
from math_verify import LatexExtractionConfig, parse, verify
from latex2sympy2_extended import NormalizationConfig
from typing import Optional

def accuracy_reward(completions: list[list[dict[str, str]]], solution: list[str], **kwargs) -> list[Optional[float]]:
    """Reward function that checks if the completion matches the ground truth.
    - If both gold and prediction are parseable → use math verification.
    - If not parseable → compare as normalized text.
    """
    rewards = []

    for completion, sol in zip(completions, solution):
        completion = _extract_text(completion)  # handle messages format
        try:
            gold_parsed = parse(sol, extraction_mode="first_match")
        except Exception as e:
            gold_parsed = []

        if len(gold_parsed) != 0:
            # Try parsing predicted answer too
            try:
                answer_parsed = parse(
                    completion,
                    extraction_config=[
                        LatexExtractionConfig(
                            normalization_config=NormalizationConfig(
                                nits=False,
                                malformed_operators=False,
                                basic_latex=True,
                                boxed="all",
                                units=True,
                            ),
                            boxed_match_priority=0,
                            try_extract_without_anchor=False,
                        )
                    ],
                    extraction_mode="first_match",
                )
                reward = float(verify(gold_parsed, answer_parsed))
            except Exception as e:
                print(f"verify failed: {e}, answer: {completion}, gold: {sol}")
                reward = None
        else:
            # fallback to text match
            reward = float(completion.strip().lower() == sol.strip().lower())

        rewards.append(reward)

    return rewards

### 3.4 Configure the GRPO training parameters

Configure the GRPO training parameters. You can vary `max_completion_length`, `num_generations`, and `max_prompt_length` to compare training configurations.

These parameter values fit the hardware limits of a Google Colab session. Larger models, more generations, and a larger, diverse dataset would allow more complete training. These changes may improve the rewards, especially for the second objective, and may improve reasoning performance.

In [ ]:
from trl import GRPOConfig

# Configure training arguments using GRPOConfig
training_args = GRPOConfig(
    output_dir="./Qwen3-VL-2B-Instruct-Thinking",
    learning_rate=1e-5,
    remove_unused_columns=False, # to access the solution column in accuracy_reward
    num_train_epochs=1,
    bf16=True,

    # Parameters that control the data preprocessing
    per_device_train_batch_size=2,
    max_completion_length=1024, # default: 256
    num_generations=2, # default: 8

    # Parameters related to reporting and saving
    report_to=["tensorboard"],
    logging_steps=10,
    push_to_hub=True,
    hub_model_id=os.environ.get("HF_ACCOUNT", "aegean-ai") + "/Qwen3-VL-2B-Instruct-Thinking",
    save_strategy="steps",
    save_steps=10,
)

### 3.5 Train the model 🏃

Configure the trainer and start training.

Pass the model, training arguments, dataset, and the two reward functions defined above to the trainer.

The diagram below shows the training procedure from the [Open-R1 project](https://github.com/huggingface/open-r1).

![Training procedure from the Open-R1 project](images/open-r1-training-procedure.png)

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
)

Train the model.

In [ ]:
trainer.train()

The training metrics are available in TensorBoard on the [model page]((https://huggingface.co/sergiopaniego/Qwen2.5-VL-3B-Instruct-Thinking/tensorboard). The loss curve may appear irregular, but the reward increases during training.

Save the results to your account 💾

In [ ]:
trainer.save_model(training_args.output_dir)
trainer.push_to_hub(dataset_name=dataset_id)

## 4. Evaluate the model

Evaluate the trained model qualitatively.

> Restart your session to release the resources used for training.

In [ ]:
import os
trained_model_id = os.environ.get("HF_ACCOUNT", "aegean-ai") + "/Qwen3-VL-2B-Instruct-Thinking"

Use the test subset for evaluation. First, load the trained model and its processor.

In [ ]:
# Reuse the trained model directly (avoids any transformers<->peft reload issues)
from transformers import AutoProcessor

trained_model = trainer.model
trained_processor = processor
print("Using trainer model for inference")

Define a helper function that accepts a problem and an image and returns the model response. The response should contain a reasoning trace and a final answer.

In [ ]:
import time
import torch
from qwen_vl_utils import process_vision_info


def generate_with_reasoning(problem, image):
    # Conversation setting for sending to the model
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": problem},
            ],
        },
    ]
    prompt = trained_processor.apply_chat_template(
      conversation,
      add_generation_prompt=True,
      tokenize=False
    )

    # Process images using the process_vision_info from qwen_vl_utils
    image_inputs, video_inputs = process_vision_info(conversation)

    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(trained_model.device)

    # Generate text without gradients
    start_time = time.time()
    with torch.no_grad():
        output_ids = trained_model.generate(**inputs, max_new_tokens=500)
    end_time = time.time()

    # Decode and extract model response
    generated_text = trained_processor.decode(output_ids[0], skip_special_tokens=True)

    # Get inference time
    inference_duration = end_time - start_time

    # Get number of generated tokens
    num_input_tokens = inputs["input_ids"].shape[1]
    num_generated_tokens = output_ids.shape[1] - num_input_tokens

    return generated_text, inference_duration, num_generated_tokens

Run the model on a test example.

In [ ]:
generated_text, inference_duration, num_generated_tokens = generate_with_reasoning(test_dataset[0]['problem'], test_dataset[0]['image'])
print(generated_text)

The answer follows the format imposed by the reward functions: `<think>reasoning</think><answer>solution</answer>`. Compare it with the reference solution to determine whether it is correct.

In [ ]:
test_dataset[0]['solution']

The model now produces a reasoning trace with its answer. Check the inference time and generated token count to further assess its performance.

In [ ]:
print(f"Inference time: {inference_duration:.2f} seconds")
print(f"Generated tokens: {num_generated_tokens}")

## 5. Further reading 🧑‍🎓

The following resources cover GRPO, reasoning, and VLMs:

* [`Post training an LLM for reasoning with GRPO in TRL` recipe and the linked resources that it includes](https://huggingface.co/learn/cookbook/fine_tuning_llm_grpo_trl)
* [GLM-4.1V-9B-Thinking paper](https://huggingface.co/papers/2507.01006)
* [TRL GRPO for VLMs example](https://github.com/huggingface/trl/examples/scripts/grpo_vlm.py)